# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pratyush457/week-1-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use a **Random Forest classifier** for the CTR / Engagement Opportunity Scoring lane.

The goal is to rank content opportunities using decision-time search and engagement signals, while allowing the model to capture non-linear relationships and interactions between signals.

Random Forest is a reasonable next step after the Week-4 rule because it can model interactions without requiring a linear relationship between the inputs and the target. It also provides feature-importance information that can be used for interpretation.

The model is not being chosen because it is more complex. It is being used because it provides a practical, interpretable comparison against the Week-4 baseline.

The model will be evaluated on the same data and metric used for the baseline, with a validation design that prevents future information from entering the decision-time features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

I will use a **time-aware split** because the task is based on making a content-prioritization decision using information available at a particular date.

The training data will use an earlier time period, while the validation data will use a later period. This prevents information from the future validation period from being used when fitting the model.

This is more honest than a random row split because the same content can appear on multiple reporting dates. A random split could therefore place observations from the same content in both training and validation data and make performance look better than it would be for a real future decision.

The split will also follow the same decision-time logic used by the Week-4 baseline so that the model and baseline are compared on the same data, target, and evaluation metric.

In [1]:
# Section 2: Split design check

print("Split design: time-aware validation")
print("Training period: earlier decision-time observations")
print("Validation period: later decision-time observations")
print("Reason: prevents future information from entering model training.")

Split design: time-aware validation
Training period: earlier decision-time observations
Validation period: later decision-time observations
Reason: prevents future information from entering model training.


## 3. Train + compare vs my baseline

The Week-4 baseline ranks March content opportunities using impressions and CTR. For this modeling step, I use the same March decision-time observations and evaluate both approaches against the same future outcome: whether the content receives clicks in April.

The target is derived only from the future outcome and is not included as a model input. Model features are limited to signals available at the March decision point.

I use a Random Forest classifier because it can capture non-linear relationships between the available search signals while remaining interpretable through feature importance.

The comparison uses Precision@50: among the top 50 ranked recommendations, how many correspond to content that receives at least one click in the future evaluation window.

The model is treated as decision-support rather than a causal predictor. A higher score does not prove that changing a page will cause more clicks.

In [5]:
# Section 3: Train + compare vs Week-4 baseline
# RAM-conscious version

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# 1. Build March decision-time features
# ---------------------------------------------------------

march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE
            WHEN gsc_impressions > 0
            THEN 100.0 * gsc_clicks / gsc_impressions
            ELSE 0
        END AS ctr_pct
    FROM {march}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_impressions > 0
""").df()

# ---------------------------------------------------------
# 2. April future-click target
# ---------------------------------------------------------

april = f"""
read_parquet(
    '{base}/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

future_clicks = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS future_clicks
    FROM {april}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

future_clicks["target"] = (
    future_clicks["future_clicks"] > 0
).astype("int8")

# ---------------------------------------------------------
# 3. Join March features with future target
# ---------------------------------------------------------

model_df = march_features.merge(
    future_clicks[
        [
            "client_hash_id",
            "content_hash_id",
            "future_clicks",
            "target"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["report_date"] = pd.to_datetime(
    model_df["report_date"]
)

print("Rows available for modeling:", len(model_df))
print("Positive targets:", int(model_df["target"].sum()))

# ---------------------------------------------------------
# 4. Time-aware split
# ---------------------------------------------------------

split_date = model_df["report_date"].quantile(0.80)

train_df = model_df[
    model_df["report_date"] < split_date
].copy()

valid_df = model_df[
    model_df["report_date"] >= split_date
].copy()

print("Training rows:", len(train_df))
print("Validation rows:", len(valid_df))
print("Split date:", split_date)

# ---------------------------------------------------------
# 5. Features available at decision time
# ---------------------------------------------------------

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr_pct"
]

# ---------------------------------------------------------
# 6. Controlled training sample
#    Keeps Random Forest practical on Colab
# ---------------------------------------------------------

MAX_TRAIN_ROWS = 300_000

if len(train_df) > MAX_TRAIN_ROWS:
    train_sample = train_df.sample(
        n=MAX_TRAIN_ROWS,
        random_state=42
    )
else:
    train_sample = train_df

X_train = train_sample[feature_cols].fillna(0)
y_train = train_sample["target"]

X_valid = valid_df[feature_cols].fillna(0)
y_valid = valid_df["target"]

print("Rows used to train model:", len(train_sample))

# ---------------------------------------------------------
# 7. Train Random Forest
# ---------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=60,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

model.fit(X_train, y_train)

# ---------------------------------------------------------
# 8. Model ranking
# ---------------------------------------------------------

valid_df["model_score"] = model.predict_proba(
    X_valid
)[:, 1]

# ---------------------------------------------------------
# 9. Week-4 baseline score
#    EXACT Week-4 formula
# ---------------------------------------------------------

valid_df["baseline_score"] = (
    np.log1p(valid_df["gsc_impressions"])
    * (
        1 -
        valid_df["ctr_pct"].clip(
            lower=0,
            upper=100
        ) / 100
    )
)

# ---------------------------------------------------------
# 10. Precision@50
# ---------------------------------------------------------

def precision_at_50(df, score_col):
    top50 = df.nlargest(50, score_col)
    return top50["target"].mean()

baseline_p50 = precision_at_50(
    valid_df,
    "baseline_score"
)

model_p50 = precision_at_50(
    valid_df,
    "model_score"
)

# ---------------------------------------------------------
# 11. Model vs baseline
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "approach": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

print("\nMODEL VS BASELINE")
display(comparison)

print(
    f"Baseline Precision@50: {baseline_p50:.3f}"
)

print(
    f"Random Forest Precision@50: {model_p50:.3f}"
)

# ---------------------------------------------------------
# 12. Feature importance
# ---------------------------------------------------------

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nFEATURE IMPORTANCE")
display(feature_importance)

# ---------------------------------------------------------
# 13. Top model predictions for later error analysis
# ---------------------------------------------------------

top_model = valid_df.nlargest(
    20,
    "model_score"
)[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "future_clicks",
        "target",
        "model_score",
        "baseline_score"
    ]
]

print("\nTOP 20 MODEL RECOMMENDATIONS")
display(top_model)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows available for modeling: 3536466
Positive targets: 1617502
Training rows: 2794961
Validation rows: 741505
Split date: 2026-03-26 00:00:00
Rows used to train model: 300000

MODEL VS BASELINE


,approach,Precision@50
0,Week-4 baseline,0.86
1,Random Forest,1.00


Baseline Precision@50: 0.860
Random Forest Precision@50: 1.000

FEATURE IMPORTANCE


,feature,importance
0,gsc_impressions,0.760313
3,ctr_pct,0.097216
2,gsc_avg_position,0.073220
1,gsc_clicks,0.069251



TOP 20 MODEL RECOMMENDATIONS


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr_pct,future_clicks,target,model_score,baseline_score
2738536,client_73cda7b4e4f265ea,content_6f58963f5afe9f5c,2026-03-27,1676,10,0.596659,208.0,1,0.996548,7.380461
3204332,client_73cda7b4e4f265ea,content_72b8a178d1f2aa59,2026-03-30,1687,10,0.592768,218.0,1,0.996548,7.387249
3336002,client_62f4a7e64f5e0096,content_922f95c9418ff0ec,2026-03-28,1668,10,0.599520,277.0,1,0.996548,7.375496
3432777,client_62f4a7e64f5e0096,content_70d006fa5dc92f80,2026-03-31,1829,11,0.601422,258.0,1,0.996548,7.466892
2808663,client_73cda7b4e4f265ea,content_06f0cf40fe8a535f,2026-03-26,1907,11,0.576822,364.0,1,0.996499,7.510239
2960953,client_73cda7b4e4f265ea,content_44aa58c15213aa83,2026-03-27,1897,11,0.579863,278.0,1,0.996499,7.504785
3198424,client_20259bd6705d81d4,content_1dbd310eb57d2182,2026-03-29,1891,11,0.581703,1131.0,1,0.996499,7.501498
3291144,client_73cda7b4e4f265ea,content_3f63244766ce99bd,2026-03-28,1746,10,0.572738,341.0,1,0.996499,7.422897
3397843,client_73cda7b4e4f265ea,content_06f0cf40fe8a535f,2026-03-31,1898,11,0.579557,364.0,1,0.996499,7.505331
3388267,client_62f4a7e64f5e0096,content_e2973b33bcc91ac1,2026-03-27,1921,10,0.520562,189.0,1,0.996487,7.521761


In [3]:
# Setup: Hugging Face + DuckDB connection

%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

base = "hf://datasets/FlyRank/internship-warehouse"

march = f"""
read_parquet(
    '{base}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("DuckDB connection ready.")
print("March data source ready.")

DuckDB connection ready.
March data source ready.


## 4. Errors and interpretation

The Random Forest achieved higher Precision@50 than the Week-4 baseline on the same validation rows, but the result should be interpreted as a measured ranking result rather than proof of future performance in all settings.

The model relies most strongly on GSC impressions, followed by CTR, average position, and clicks. This indicates that visibility is the strongest observed input in this model.

The top-ranked validation examples were mostly positive under the defined future-click target. However, false positives can still occur when a page receives high predicted opportunity but does not receive future clicks. These cases may reflect changes in search demand, page intent, competition, or other factors not represented by the available features.

The model therefore supports prioritization and human review rather than automatic content changes. The Precision@50 result is directional evidence from this validation setup, not a causal claim that changing a page will increase clicks.

In [6]:
# Section 4: Error analysis

valid_df["predicted_positive"] = (
    valid_df["model_score"] >= 0.5
).astype(int)

# False positives: high model score but no future clicks
false_positives = valid_df[
    valid_df["target"] == 0
].nlargest(
    10,
    "model_score"
)

# False negatives: future clicks occurred but model score was relatively low
false_negatives = valid_df[
    valid_df["target"] == 1
].nsmallest(
    10,
    "model_score"
)

print("FALSE POSITIVES — high model score but no future clicks")
display(
    false_positives[
        [
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "ctr_pct",
            "model_score",
            "future_clicks",
            "target"
        ]
    ]
)

print("\nFALSE NEGATIVES — future clicks but lower model score")
display(
    false_negatives[
        [
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "ctr_pct",
            "model_score",
            "future_clicks",
            "target"
        ]
    ]
)

print("\nModel interpretation:")
print("The strongest feature by Random Forest importance was:",
      feature_importance.iloc[0]["feature"])

FALSE POSITIVES — high model score but no future clicks


,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr_pct,model_score,future_clicks,target
3072918,content_844c058f46644c1f,2026-03-27,740,6,0.810811,0.995043,0.0,0
3509340,content_49fcf8ec7ec942ff,2026-03-27,480,2,0.416667,0.994858,0.0,0
3379880,content_844c058f46644c1f,2026-03-26,563,4,0.710480,0.994662,0.0,0
3457987,content_844c058f46644c1f,2026-03-28,1638,5,0.305250,0.994367,0.0,0
2598354,content_735904e3579013d0,2026-03-26,378,2,0.529101,0.994247,0.0,0
3361332,content_1f9225afeca5a808,2026-03-29,407,2,0.491400,0.994209,0.0,0
3147519,content_24f01132527241cf,2026-03-28,501,2,0.399202,0.994206,0.0,0
2728482,content_4c163d9bcbe54fb3,2026-03-26,720,2,0.277778,0.994203,0.0,0
2911794,content_413865fe64925077,2026-03-26,744,2,0.268817,0.994203,0.0,0
2801623,content_33184ef3a572613d,2026-03-26,345,2,0.579710,0.994145,0.0,0



FALSE NEGATIVES — future clicks but lower model score


,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr_pct,model_score,future_clicks,target
2719653,content_ec4c376fa3245931,2026-03-26,2,0,0.0,0.061422,1.0,1
2813735,content_20249d88571b82e9,2026-03-26,2,0,0.0,0.061422,1.0,1
2820816,content_d3ea844aa1747c18,2026-03-26,2,0,0.0,0.061422,1.0,1
2850374,content_433e020ddeb5e601,2026-03-26,2,0,0.0,0.061422,2.0,1
2881314,content_94930c80edc4795b,2026-03-29,2,0,0.0,0.061422,1.0,1
2936883,content_d0851703106cdb11,2026-03-27,2,0,0.0,0.061422,1.0,1
2983637,content_db6d709d8242da58,2026-03-28,2,0,0.0,0.061422,1.0,1
3122344,content_56b233a561a8f23b,2026-03-27,2,0,0.0,0.061422,2.0,1
3268660,content_18eb04ab7a47b156,2026-03-31,2,0,0.0,0.061422,1.0,1
3449163,content_73f56d7c617d6b7b,2026-03-28,2,0,0.0,0.061422,1.0,1



Model interpretation:
The strongest feature by Random Forest importance was: gsc_impressions


## Self-check

Before submitting, I confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] The model is compared with the Week-4 baseline using the same validation data and Precision@50 metric.
- [x] The validation design is time-aware and uses decision-time features only.
- [x] Future clicks are used only to define the evaluation target, not as model inputs.
- [x] Feature importance and model errors were reviewed.
- [x] The notebook is committed to my repo under `work/notebooks/w05_model.ipynb`.